# Relación entre estaciones meteorológicas y estaciones de fenología

Los archivos de `datos/raw/estaciones/*.XLS` son exportaciones de la base de SENAMHI (tablas HTML con extensión .XLS)
con datos diarios de **PT101** (precipitación, mm), **TM102** (temperatura máxima, °C) y **TM103** (temperatura mínima, °C).
El nombre de la estación solo está en el **nombre del archivo**; adentro viene el código de la estación.

Este notebook:
1. Lee los archivos de clima y resume cada estación (código, período, completitud).
2. Empareja el nombre del archivo con las estaciones de la base de fenología consolidada.
3. Revisa los emparejamientos dudosos (nombres repetidos, renombres, períodos que no coinciden).
4. Guarda la tabla de relación y una base diaria de clima unida a los nombres de fenología.

**Salidas** (en `datos/processed/`):
- `relacion_estaciones.xlsx` / `.csv`: una fila por estación climática y su(s) nombre(s) en fenología.
- `clima_estaciones_diario.csv`: datos diarios de todas las estaciones, con la columna `estacion_fenologia`.

In [ ]:
import difflib
import hashlib
import re
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "scripts"))
from consolidar_fenologia import sin_tildes  # misma normalización que la base consolidada

DIR_CLIMA = RAIZ / "datos" / "raw" / "estaciones"
FENOLOGIA = RAIZ / "datos" / "processed" / "FENOLOGIA_consolidado.csv"
SALIDA = RAIZ / "datos" / "processed"
FALTANTE = -999          # valor de dato faltante en los archivos de SENAMHI

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)

## 1. Archivos de clima

In [ ]:
VARIABLES = {"PT101": "precipitacion", "TM102": "tmax", "TM103": "tmin"}

def leer_clima(ruta):
    """Tabla HTML de SENAMHI -> DataFrame diario con fecha, precipitacion, tmax, tmin."""
    t = pd.read_html(ruta, encoding="latin-1")[0]
    fila_enc = t.index[t.iloc[:, 0].astype(str).str.strip() == "Codigo"][0]
    d = t.loc[fila_enc + 1:].copy()
    d.columns = [str(c).strip() for c in t.loc[fila_enc]]
    d = d.rename(columns={"Año": "anio", "Mes": "mes", "V Dia": "dia", "Codigo": "codigo"})
    d["fecha"] = pd.to_datetime(dict(year=d["anio"].astype(int), month=d["mes"].astype(int),
                                     day=d["dia"].astype(int)), errors="coerce")
    for col, nombre in VARIABLES.items():
        v = pd.to_numeric(d[col], errors="coerce")
        d[nombre] = v.mask(v <= FALTANTE + 0.5)          # -999 -> faltante
    d["codigo"] = d["codigo"].astype(str).str.zfill(6)
    return d[["codigo", "fecha", *VARIABLES.values()]].dropna(subset=["fecha"])


archivos = sorted(DIR_CLIMA.glob("*.XLS"))
clima, resumen = {}, []
for ruta in archivos:
    d = leer_clima(ruta)
    clima[ruta.name] = d
    dias = (d["fecha"].max() - d["fecha"].min()).days + 1
    resumen.append({
        "archivo": ruta.name,
        "md5": hashlib.md5(ruta.read_bytes()).hexdigest(),
        "codigo": d["codigo"].iloc[0],
        "codigos_distintos": d["codigo"].nunique(),
        "desde": d["fecha"].min().date(), "hasta": d["fecha"].max().date(),
        "dias_periodo": dias,
        **{f"pct_{v}": round(100 * d[v].notna().sum() / dias, 1) for v in VARIABLES.values()},
    })
resumen = pd.DataFrame(resumen)
print(f"{len(archivos)} archivos leídos")
resumen

**Archivos duplicados**: mismo contenido (md5) o mismo código de estación. Se conserva uno por código.

In [ ]:
dup = resumen[resumen.duplicated("codigo", keep=False)].sort_values("codigo")
display(dup[["archivo", "codigo", "md5", "desde", "hasta"]])
print("¿mismo contenido?", dup.groupby("codigo")["md5"].nunique().eq(1).to_dict())

## 2. Estaciones de la base de fenología

In [ ]:
fen = pd.read_csv(FENOLOGIA, dtype={"ubigeo": str}, low_memory=False,
                  usecols=["estacion", "estacion_norm", "ddrr", "zona", "departamento", "provincia",
                           "distrito", "ubigeo", "cultivo_norm", "semana_inicio", "tipo_registro"])

def moda(s):
    s = s.dropna()
    return s.mode().iloc[0] if len(s) else None

# La base trae ddrr_norm (DDRR corregida); se recalcula aquí igual para mostrar los homónimos.
# Algunas filas traen la DDRR mal escrita ("VFIF", "7"...) o arrastrada de otro bloque: se ignoran
# las DDRR con menos del 10 % de los registros de la estación. Si quedan dos o más, son homónimos
# reales (lugares distintos con el mismo nombre) y se tratan como estaciones separadas.
MIN_PCT_DDRR = 10
cuenta = fen.groupby(["estacion_norm", "ddrr"]).size().rename("n").reset_index()
cuenta["pct"] = 100 * cuenta["n"] / cuenta.groupby("estacion_norm")["n"].transform("sum")
validas = cuenta[cuenta["pct"] >= MIN_PCT_DDRR][["estacion_norm", "ddrr"]]
homonimos = validas.groupby("estacion_norm").size().loc[lambda s: s > 1].index.tolist()
print("Homónimos (mismo nombre en varias DDRR):", homonimos)
display(cuenta[cuenta["estacion_norm"].isin(homonimos)])

# Filas de ruido -> se reasignan a la DDRR principal de la estación (salvo homónimos)
principal = cuenta.sort_values("n").groupby("estacion_norm")["ddrr"].last()
fen["ddrr_est"] = fen["ddrr"]
ruido = ~fen.set_index(["estacion_norm", "ddrr"]).index.isin(validas.set_index(["estacion_norm", "ddrr"]).index)
fen.loc[ruido, "ddrr_est"] = fen.loc[ruido, "estacion_norm"].map(principal)

est_fen = fen.groupby(["estacion_norm", "ddrr_est"]).agg(
    nombres_originales=("estacion", lambda s: " | ".join(sorted(s.dropna().unique())[:5])),
    departamento=("departamento", moda),
    provincia=("provincia", moda),
    distrito=("distrito", moda),
    ubigeo=("ubigeo", moda),
    fen_desde=("semana_inicio", "min"),
    fen_hasta=("semana_inicio", "max"),
    registros=("estacion", "size"),
    cultivos=("cultivo_norm", lambda s: ", ".join(s.value_counts().index[:5])),
    registros_maiz=("cultivo_norm", lambda s: s.fillna("").str.contains("maiz").sum()),
).reset_index().rename(columns={"ddrr_est": "ddrr"})
est_fen["homonimo"] = est_fen["estacion_norm"].isin(homonimos)

# Nombre base: sin prefijos de tipo de estación (CO-, CP-, MAP-, ...) ni marcas como "(*)"
PREFIJOS = r"^(CO|CP|PE|PLU|MAP|HLM|HLG)-\s*"
est_fen["nombre_base"] = (est_fen["estacion_norm"]
                          .str.replace(PREFIJOS, "", regex=True)
                          .str.replace(r"\(\*\)", "", regex=True)
                          .str.strip())
print(f"{len(est_fen)} estaciones en la base de fenología")
est_fen.head()

## 3. Emparejamiento por nombre

1. **exacto**: el nombre del archivo (sin tildes, en mayúsculas) es igual al nombre base de fenología.
2. **manual**: correcciones de nombres de archivo mal escritos (diccionario `MANUAL`, editable).
3. **similar**: el nombre más parecido con `difflib` (similitud ≥ 0,8). Queda marcado para revisión.

Una estación climática puede corresponder a **varios nombres** en fenología cuando la estación se renombró
(p. ej. `SAN MARCOS` → `CO-SAN MARCOS`, `CACHACHI` → `PE-CACHACHI`): se asignan todos los que comparten el nombre base.

In [ ]:
# Nombre de archivo -> nombre base en fenología (para typos en los nombres de archivo)
MANUAL = {
    "CANADARAVE": "CANDARAVE",
    "MACHAHUAY": "MACHAGUAY",
    "SANTO DOMINGDZ1_O": "SANTO DOMINGO",   # 'dz1' = Dirección Zonal 1 (Piura)
    "WEBERBAUER": "A. WEBERBAUER",
    "CAICAY": "CAYCAY",                    # en fenología se unificó CO-CAICAY -> CO-CAYCAY
}
# Para estaciones con homónimos: DDRR a la que corresponde cada archivo de clima
DDRR_ARCHIVO = {
    "san pablo.XLS": "CAJAMARCA",          # se descargó junto a las de Cajamarca; la de San Martín es de arroz
    "santo domingdz1_o.XLS": "PIURA",      # 'dz1' en el nombre = Dirección Zonal 1 (Piura)
}
UMBRAL = 0.8

bases = sorted(est_fen["nombre_base"].unique())

def emparejar(nombre_archivo):
    n = sin_tildes(Path(nombre_archivo).stem).upper().strip()
    if n in bases:
        return n, n, "exacto", 1.0
    if n in MANUAL:
        return n, MANUAL[n], "manual", round(difflib.SequenceMatcher(None, n, MANUAL[n]).ratio(), 2)
    cand = difflib.get_close_matches(n, bases, n=1, cutoff=UMBRAL)
    if cand:
        return n, cand[0], "similar (revisar)", round(difflib.SequenceMatcher(None, n, cand[0]).ratio(), 2)
    return n, None, "sin coincidencia", None

m = resumen[["archivo", "codigo"]].copy()
m[["nombre_archivo", "nombre_base", "metodo", "similitud"]] = m["archivo"].apply(
    lambda a: pd.Series(emparejar(a)))
m["metodo"].value_counts()

In [ ]:
# Unir con todos los nombres de fenología que comparten el nombre base
rel = m.merge(est_fen, on="nombre_base", how="left")
# En homónimos, quedarse solo con la DDRR indicada para el archivo
elegida = rel["archivo"].map(DDRR_ARCHIVO)
rel = rel[~rel["homonimo"].fillna(False) | (rel["ddrr"] == elegida)]
sin_ddrr = rel[rel["homonimo"].fillna(False) & elegida.reindex(rel.index).isna()]
assert sin_ddrr.empty, f"Falta DDRR_ARCHIVO para: {sin_ddrr['archivo'].unique()}"

# Período en común entre el clima y la fenología
per = resumen.set_index("archivo")[["desde", "hasta", "pct_tmax", "pct_tmin", "pct_precipitacion"]]
rel = rel.join(per, on="archivo")
rel["fen_desde"] = pd.to_datetime(rel["fen_desde"]).dt.date
rel["fen_hasta"] = pd.to_datetime(rel["fen_hasta"]).dt.date
# (archivos sin estación de fenología quedan con fechas vacías)
ini = rel[["desde", "fen_desde"]].apply(pd.to_datetime).max(axis=1, skipna=False)
fin = rel[["hasta", "fen_hasta"]].apply(pd.to_datetime).min(axis=1, skipna=False)
rel["dias_en_comun"] = (fin - ini).dt.days.clip(lower=0)

# Alertas para revisar
def alertas(r):
    a = []
    if r["metodo"] != "exacto":
        a.append(f"emparejado por {r['metodo']}")
    if pd.isna(r["estacion_norm"]):
        a.append("sin estación de fenología")
        return "; ".join(a)
    if r["homonimo"]:
        a.append(f"nombre repetido en otra DDRR; se asignó {r['ddrr']} (confirmar con el código)")
    if r["n_nombres_fen"] > 1:
        a.append(f"{r['n_nombres_fen']} nombres en fenología (renombre o duplicado)")
    if r["dias_en_comun"] == 0:
        a.append("sin período en común entre clima y fenología")
    return "; ".join(a) or None

rel = rel.reset_index(drop=True)
rel["n_nombres_fen"] = rel.groupby("archivo")["estacion_norm"].transform("count")
rel["revisar"] = rel.apply(alertas, axis=1)
rel["duplicado_de"] = rel["codigo"].map(
    resumen.drop_duplicates("codigo").set_index("codigo")["archivo"])
rel.loc[rel["duplicado_de"] == rel["archivo"], "duplicado_de"] = None

cols = ["archivo", "codigo", "nombre_archivo", "metodo", "similitud", "estacion_norm",
        "nombres_originales", "ddrr", "departamento", "provincia", "distrito", "ubigeo",
        "desde", "hasta", "fen_desde", "fen_hasta", "dias_en_comun",
        "pct_tmax", "pct_tmin", "pct_precipitacion", "registros", "registros_maiz", "cultivos",
        "duplicado_de", "revisar"]
rel = rel[cols].rename(columns={"estacion_norm": "estacion_fenologia",
                                "desde": "clima_desde", "hasta": "clima_hasta"})
rel

### Casos para revisar

In [ ]:
rel[rel["revisar"].notna()][["archivo", "codigo", "estacion_fenologia", "ddrr", "fen_desde", "fen_hasta",
                             "registros", "revisar"]]

Notas sobre los casos marcados:
- **Renombres**: la base consolidada ya unifica las variantes de nombre de una misma estación
  (`ESTACION_TYPOS` en `scripts/consolidar_fenologia.py`; p. ej. `SAN MARCOS` → `CO-SAN MARCOS`,
  `CACHACHI` → `PE-CACHACHI`). Si aparece "varios nombres en fenología", falta agregar una equivalencia allí.
- **Homónimos**: `CO-SAN PABLO` (Cajamarca, maíz / San Martín, arroz) y `SANTO DOMINGO` (Lima, manzano / Piura, maíz)
  son lugares distintos con el mismo nombre. La DDRR de cada archivo se fija en `DDRR_ARCHIVO`; conviene confirmarla con
  el código de la estación climática. En los análisis por estación, usar la clave **estación + DDRR**.
- Los datos de **departamento/provincia/distrito** son los más frecuentes de la base de fenología, que tiene algunas
  filas desalineadas; el código de estación es la referencia más confiable.

## 4. Estaciones de fenología sin archivo de clima

In [ ]:
con_clima = set(zip(rel["estacion_fenologia"], rel["ddrr"]))
sin = (est_fen[[k not in con_clima for k in zip(est_fen["estacion_norm"], est_fen["ddrr"])]]
       .sort_values(["registros_maiz", "registros"], ascending=False)
       [["estacion_norm", "ddrr", "departamento", "fen_desde", "fen_hasta", "registros",
         "registros_maiz", "cultivos"]])
print(f"{len(con_clima)} estaciones de fenología con clima | {len(sin)} sin archivo de clima")
# Estaciones de lista_estaciones.txt (si existe) que no tienen archivo de clima
lista = SALIDA / "lista_estaciones.txt"
if lista.exists():
    pedidas = [l.strip() for l in lista.read_text(encoding="utf-8").splitlines() if l.strip()]
    faltan = [e for e in pedidas if e not in {k[0] for k in con_clima}]
    print(f"lista_estaciones.txt: {len(pedidas)} estaciones, {len(faltan)} sin archivo de clima:")
    print(", ".join(faltan))
sin.head(30)

## 5. Guardar resultados

In [ ]:
rel.to_csv(SALIDA / "relacion_estaciones.csv", index=False, encoding="utf-8-sig")
with pd.ExcelWriter(SALIDA / "relacion_estaciones.xlsx") as xw:
    rel.to_excel(xw, sheet_name="relacion", index=False)
    resumen.drop(columns="md5").to_excel(xw, sheet_name="archivos_clima", index=False)
    sin.to_excel(xw, sheet_name="fenologia_sin_clima", index=False)

# Base diaria de clima: un archivo por código (sin duplicados), con el/los nombres de fenología
nombres = (rel[rel["duplicado_de"].isna()]
           .groupby("archivo")
           .agg(estacion_fenologia=("estacion_fenologia", lambda s: " | ".join(s.dropna())),
                ddrr=("ddrr", "first")))
diario = pd.concat(
    [clima[a].assign(archivo=a, estacion_fenologia=nombres.at[a, "estacion_fenologia"] or None,
                     ddrr=nombres.at[a, "ddrr"])
     for a in rel.loc[rel["duplicado_de"].isna(), "archivo"].unique()],
    ignore_index=True)
diario = diario[["codigo", "archivo", "ddrr", "estacion_fenologia", "fecha", "precipitacion", "tmax", "tmin"]]
diario.to_csv(SALIDA / "clima_estaciones_diario.csv", index=False, encoding="utf-8-sig")
print(f"relacion_estaciones: {len(rel)} filas | clima diario: {len(diario):,} filas de "
      f"{diario['codigo'].nunique()} estaciones")